# 1. Explore your group’s local data partition

## Goal

In this notebook, you will examine the data available to **your group**
before federated learning begins.

Your group has one local partition. Other groups’ partitions are not
mounted in this workspace and are not inspected here.

**Run the cells from top to bottom.** The workshop software is already
prepared before this notebook opens; you do not need to install anything.


## What you will learn

By the end, you should be able to explain:

1. which group you represent;
2. how many examples are in your local partition;
3. how labels are distributed in that partition; and
4. why the tutorial uses synthetic features rather than opening image files.


In [ ]:
import os
import sys
from pathlib import Path

import flwr
import pandas as pd

group_id = os.environ["GROUP_ID"]
workshop_root = Path(os.environ["DIGITAFRICA_WORKSHOP_ROOT"])
data_path = Path(os.environ["CLIENT_DATA_PATH"])
app_root = workshop_root / "app"

if not group_id.startswith("group_"):
    raise RuntimeError(
        f"This notebook requires a workshop group login, got {group_id!r}."
    )
if not data_path.is_file():
    raise FileNotFoundError(f"Local partition is unavailable: {data_path}")
if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

print("Workshop environment is ready.")
print(f"Your group:          {group_id}")
print(f"Your local partition: {data_path.name}")
print(f"Flower version:       {flwr.__version__}")
print("Only your group's prepared CSV metadata is available in this notebook.")


## Step 1 — Read local metadata

Each row represents one example. The `image` value is an identifier;
this tutorial does **not** open the source image file. The `label` value
is the class associated with that example.


In [ ]:
partition = pd.read_csv(data_path)

required_columns = {"image", "label"}
missing_columns = required_columns.difference(partition.columns)
if missing_columns:
    raise ValueError(
        f"Partition is missing required columns: {sorted(missing_columns)}"
    )

print(f"Examples in {group_id}'s local partition: {len(partition)}")
print("\nClass distribution:")
display(
    partition["label"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .to_frame("examples")
)

print("\nFirst five metadata rows:")
display(partition.head())


## Step 2 — Prepare a privacy-preserving workflow demonstration

A real image model would need access to image pixels and an approved
modelling protocol. This workshop demonstrates the *federated-learning
workflow* instead.

It derives deterministic synthetic numeric features from local metadata.
This lets every group participate in the same protocol without opening
source image files or moving local rows to the central server.


In [ ]:
from client.client import load_partition

features, labels = load_partition(
    data_path,
    num_classes=5,
    feature_dim=16,
    seed=42,
)

print("Synthetic representation prepared successfully.")
print(f"Feature matrix: {features.shape[0]} local examples × {features.shape[1]} features")
print(f"Label vector:   {labels.shape[0]} local labels")
print(f"Feature type:   {features.dtype}")
print(
    "\nThe feature matrix remains inside this group workspace. "
    "Only model updates are exchanged during federation."
)


## Ready for federation

You have now checked the local partition used by your group.

**Do not start a client yet.** Wait for the organiser to confirm that all
groups are ready. Then open `02_Run_Federated_Client.ipynb`.

**Check your understanding:** Why is it useful that each group can inspect
its own label distribution before training starts?
